# HanziGen - 字型生成训练（腾讯云 Cloud Studio · 三阶段机时优化版）

## 使用前准备

1. **创建 Cloud Studio 工作空间**：访问 [cloudstudio.net](https://cloudstudio.net)，选择「**All in One**」空白模板
2. **上传本 notebook** 至工作空间根目录
3. **上传目标字体**（`.ttf` 或 `.otf`，文件名用英文）到 `fonts/` 目录
4. 按下方「三阶段硬件策略」选择实例，从 Cell 0 开始按顺序运行

---

## 三阶段硬件策略（压榨机时性价比的核心）

| 阶段 | 内容 | 推荐实例 | 理由 |
|---|---|---|---|
| 一、数据准备 | Cell 0-2：覆盖率分析、渲染字形图、划分字集 | 最便宜 CPU（1核2G 起步，0.1-0.5 机时/时） | 纯文件管理与字体渲染，不需要 GPU |
| 二、模型训练 | Cell 0, 1, 3, 4：VQ-VAE + LDM | GPU：**V100 首选**（900GB/s 带宽）；A10 次之；T4 省钱但慢 | 训练需高频读写海量潜在特征，带宽决定显卡是否"算个不停" |
| 三A、推理+指标 | Cell 0, 1, 5：生成缺失字形 PNG | GPU：**T4 即可**（最便宜 GPU） | 推理为批量计算，对带宽不敏感，无需 V100 |
| 三B、导出+下载 | Cell 0, 6-8：SVG 向量化、打包下载 | CPU：**16核32G 首选**（2 机时/时）；8核16G 备选 | 1万+ 小文件写入，瓶颈在内存缓存；8G 以下内存必崩 |

> ⚠️ **每次切换实例后，务必重跑 Cell 0 + Cell 1**：Cell 1 会重新检测硬件并自动调优 `BATCH_SIZE` / `NUM_WORKERS` 等关键参数，并完成环境自检（已就绪部分自动跳过，幂等）。

---

## GPU 检测代码的调整说明（相对旧版）

旧版 Cell 1 内置 `assert torch.cuda.is_available()` 硬检查，导致 CPU 实例（数据准备 / SVG 导出阶段）无法完成初始化。新版拆成两级：

- **Cell 1 → 软检测**：无 GPU 时仅提示"当前实例可运行哪些阶段"，**不报错**；同时自动把 `extract_charset.sh` 的 `DEVICE` 改写为 `cpu`（该步仅划分数据集，无实际张量计算）
- **Cell 3 / 4 / 5 → 硬断言**：各自开头 `assert` GPU，误在 CPU 实例运行会得到明确的切换指引

---

## 流程总览

```
Cell 0: 配置字体名 + 阶段开关 + 补字基准 + 性能档位        【任意实例】
Cell 1: 环境初始化 + 硬件自动调优 + 字体切换检测 + 断连自检 【任意实例，可重复运行】
Cell 2: 数据准备（分析 → 渲染 → 划分字集）                 【CPU 实例即可】
Cell 3: 训练 VQ-VAE（自动精确续训）                        【GPU: V100 推荐】
Cell 4: 训练 LDM（自动精确续训）                           【GPU: V100 推荐】
Cell 4-前置操作: 离线放置 VGG16 权重（手动上传pth，LDM训练与指标前置）【任意实例，联网下载慢时运行】
Cell 5: 推理生成 + 评估指标（支持 jf7000/unihan/gbk 基准）  【GPU: T4 即可】
Cell 6: SVG 向量化 + 产出总览                              【CPU: 16核32G 推荐】
Cell 7: 打包下载 SVG zip                                   【CPU: 16核32G 推荐】
Cell 8: 打包下载全部产出（PNG + SVG）                      【CPU: 16核32G 推荐】
```

---
## Cell 0: 配置参数

> **只改这里！** 填你放在 `fonts/` 目录下的字体文件名（英文，不含中文与空格）。

In [ ]:
# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "your_font.otf"    # 改成你放在 fonts/ 目录下的字体名
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]

# ==================== 训练阶段开关（默认全开）====================
# 置 False 可跳过某阶段。断连恢复时无需手动设置，已完成阶段会自动跳过。
DO_DATA_PREP   = True     # Cell 2: 数据准备（CPU 实例即可）
DO_TRAIN_VQVAE = True     # Cell 3: 训练 VQ-VAE（GPU 实例）
DO_TRAIN_LDM   = True     # Cell 4: 训练 LDM（GPU 实例）
DO_INFERENCE   = True     # Cell 5: 推理 + 指标（GPU 实例）
# ==========================================================

# ==================== 补字基准（Cell 5 推理阶段生效）====================
# "jf7000" = jf7000 当务字集缺失字（默认，约 8,349 字基准，项目原生设计）
# "unihan" = Unihan 全字集缺失字（9 万+ 字基准，生成量大，注意机时）
# "gbk"    = GBK 简体标准字符集缺失字（20,902 字基准，简体用户推荐）
# "gb2312" = 仅 GB2312 简体核心字（6,763 字基准，范围最保守）
CHARSET_BASE = "jf7000"
# ==========================================================

# ==================== 硬件性能档位 ====================
# "auto" = 自动检测当前实例 GPU 并应用推荐参数（推荐）
# "v100" / "t4" / "cpu" = 手动指定档位（检测失效或想自定义时用）
HW_PROFILE = "auto"
# ==========================================================

STATE_FILE = "colab_state.json"

print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"阶段开关: 数据准备={DO_DATA_PREP} VQVAE={DO_TRAIN_VQVAE} LDM={DO_TRAIN_LDM} 推理={DO_INFERENCE}")
print(f"补字基准: {CHARSET_BASE} | 性能档位: {HW_PROFILE}")

---
## Cell 1: 全自动环境初始化 + 硬件调优 + 断连自检

> 克隆仓库 → 安装依赖 → 检查字体 → 下载 Jigmo → 改写脚本 → **字体切换检测** → **GPU 软检测 + 参数自动调优**
>
> 可在**任何实例**重复运行：已就绪的部分自动跳过；无 GPU **不会报错**（详见顶部「GPU 检测代码的调整说明」）。
>
> **换实例后必跑本 Cell**：它按当前 GPU 型号自动改写 `BATCH_SIZE` / `NUM_WORKERS` / 评估批大小，并处理续训路径。

In [ ]:
import os, shutil, json, re, sys, glob, zipfile, io, urllib.request, subprocess
from fontTools.ttLib import TTFont

# ===== 0. 克隆项目代码（首次运行时自动拉取）=====
REPO_URL  = "https://github.com/ICW-k/HanziGen_ICWfork.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

is_project_root = os.path.isdir("scripts") and os.path.exists("requirements.txt")

if not is_project_root:
    found_repo = None
    for search_root in [os.getcwd(), "/workspace", "/home", "/"]:
        if not os.path.isdir(search_root):
            continue
        try:
            for entry in os.listdir(search_root):
                candidate = os.path.join(search_root, entry)
                if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "scripts")):
                    if os.path.exists(os.path.join(candidate, "requirements.txt")):
                        found_repo = candidate
                        break
        except PermissionError:
            continue
        if found_repo:
            break

    if found_repo:
        print(f"发现已有项目目录: {found_repo}")
        os.chdir(found_repo)
    else:
        print(f"正在克隆仓库: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)
    print(f"已切换到项目根目录: {os.getcwd()}")
else:
    print("已在项目根目录，跳过克隆")

# Cloud Studio 兼容：确保 python 命令指向 python3
print("\n===== 环境兼容检查 =====")
!which python3 && ln -sf $(which python3) /usr/local/bin/python 2>/dev/null; python --version && echo "python 命令已就绪"

PROJECT = os.getcwd()
print(f"\n工作目录: {PROJECT}")
!ls -F | head -30

# ===== 1. 安装依赖 =====
print("\n===== 安装 PyTorch + 依赖 =====")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt
print("\n依赖安装完成！")

# ===== 2. 检查目标字体（智能定位）=====
print("\n===== 检查字体 =====")

def find_font_in_workspace(filename: str, search_root: str = ".") -> str | None:
    """在 search_root 下递归搜索 filename，返回第一个匹配的路径"""
    for root, dirs, files in os.walk(search_root, followlinks=False):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__pycache__", "node_modules")]
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    return None

font_path = f"fonts/{TARGET_FONT}"

print("当前 fonts/ 目录内容:")
if os.path.isdir("fonts"):
    !find fonts/ -type f 2>/dev/null | head -30
else:
    print("  [WARN] fonts/ 目录不存在！")

if not os.path.exists(font_path):
    print(f"\n[WARN] {font_path} 不存在，正在搜索整个工作区...")
    found = find_font_in_workspace(TARGET_FONT, "/workspace" if os.path.isdir("/workspace") else ".")
    if found:
        print(f"[OK] 找到字体: {found}")
        os.makedirs("fonts", exist_ok=True)
        shutil.copy2(found, font_path)
        print(f"[OK] 已复制到: {font_path}")
    else:
        !ls -la
        raise FileNotFoundError(
            f"\n字体文件 '{TARGET_FONT}' 在整个工作区都找不到！\n"
            f"\n请检查:\n"
            f"  1. 文件名是否完全一致（注意大小写）? 当前配置: '{TARGET_FONT}'\n"
            f"  2. 字体是否已上传到 Cloud Studio 工作空间?\n"
            f"  3. 上传后文件是否放在 fonts/ 子目录下?"
        )
else:
    print(f"目标字体已就绪: {font_path}")

# ===== 3. 把字体路径写进所有 .sh 脚本 =====
print("\n===== 改写脚本字体路径 =====")
for sh_file in glob.glob("scripts/*.sh"):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'fonts/[\w.-]+\.(ttf|otf)', f'fonts/{TARGET_FONT}', content)
    with open(sh_file, "w", encoding="utf-8") as f:
        f.write(content)
print("脚本字体路径已统一替换")

# ===== 4. Jigmo 参考字体：从官方 ZIP 下载 =====
print("\n===== 准备 Jigmo 参考字体 =====")

jigmo_files = ["jigmo.ttf", "jigmo2.ttf", "jigmo3.ttf"]

def download_jigmo_fonts(target_dir="fonts/jigmo"):
    os.makedirs(target_dir, exist_ok=True)
    zip_url = "https://kamichikoichi.github.io/jigmo/Jigmo-20250912.zip"
    print(f"  下载 Jigmo ZIP: {zip_url}")
    try:
        resp = urllib.request.urlopen(zip_url, timeout=30)
        data = resp.read()
        if len(data) < 10000:
            raise ValueError(f"下载数据太小 ({len(data)} bytes)")
        zf = zipfile.ZipFile(io.BytesIO(data))
    except Exception as e:
        print(f"  [ERROR] ZIP 下载失败: {e}")
        return False
    for zip_name in zf.namelist():
        basename = os.path.basename(zip_name).lower()
        if basename in jigmo_files:
            zf.extract(zip_name, target_dir)
            extracted = os.path.join(target_dir, zip_name)
            target = os.path.join(target_dir, basename)
            if extracted != target:
                if os.path.exists(target):
                    os.remove(target)
                os.rename(extracted, target)
    zf.close()
    return True

def validate_font_file(fpath):
    try:
        f = TTFont(fpath)
        if "cmap" not in f:
            return False, "缺少 cmap 表"
        return True, f"OK ({len(f.getBestCmap())} glyphs)"
    except Exception as e:
        return False, str(e)[:80]

need_download = False
for fname in jigmo_files:
    fpath = f"fonts/jigmo/{fname}"
    if os.path.exists(fpath):
        valid, msg = validate_font_file(fpath)
        if not valid:
            print(f"  [WARN] {fname} 无效 ({msg})")
            os.remove(fpath)
            need_download = True
    else:
        need_download = True

if need_download:
    if not download_jigmo_fonts():
        raise RuntimeError("Jigmo 下载失败！")
    print("Jigmo 字体下载完成，验证中...")
    for fname in jigmo_files:
        valid, msg = validate_font_file(f"fonts/jigmo/{fname}")
        print(f"    [{'OK' if valid else 'ERROR'}] {fname}: {msg}")
        if not valid:
            raise RuntimeError(f"Jigmo 字体 {fname} 验证失败！")
else:
    print("Jigmo 字体已就绪，跳过下载")

# ===== 5. 断连自检 + 字体切换检测 + 自动续训改写 =====
print("\n===== 断连自检 + 字体切换检测 =====")
os.makedirs("checkpoints", exist_ok=True)

def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_state(state: dict) -> None:
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()

# --- 字体切换检测：数据集状态与字体名绑定，防止新旧字体数据混合 ---
prev_data_font = state.get("data_font")
if prev_data_font and prev_data_font != FONT_NAME:
    print(f"  [字体切换] {prev_data_font} → {FONT_NAME}")
    print("    · Cell 2 将重新执行数据准备：prepare_dataset.py 内置清空逻辑，旧字体图像会被自动删除")
    print("    · 旧字体的 checkpoints / samples / svgs 按字体名隔离，不受影响")
    if os.path.exists(f"checkpoints/vqvae_{FONT_NAME}.pth") or os.path.exists(f"checkpoints/ldm_{FONT_NAME}.pth"):
        print(f"    · [注意] 当前字体存在历史检查点。若你曾更新过 {TARGET_FONT} 文件本身，")
        print(f"      建议删除 checkpoints/vqvae_{FONT_NAME}.pth 与 ldm_{FONT_NAME}.pth 从零训练")
    state["data_prep_done"] = False
    state["data_font"] = None

state.setdefault("font", FONT_NAME)
save_state(state)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
have_vqvae = os.path.exists(vqvae_ckpt)
have_ldm = os.path.exists(ldm_ckpt)

print(f"  数据准备 (data/): {'已完成' if state.get('data_prep_done') else '未完成'}（绑定字体: {state.get('data_font') or '无'}）")
print(f"  VQ-VAE 检查点:   {'存在: '+vqvae_ckpt if have_vqvae else '不存在'}")
print(f"  LDM 检查点:      {'存在: '+ldm_ckpt if have_ldm else '不存在'}")

# --- 自动精确续训：统一改写 RESUME_FROM（存在→指向检查点；不存在→清空） ---
# 无论是否续训都执行改写，避免脚本中残留其他字体的旧路径造成跨字体污染
def _set_resume_from(sh_path: str, ckpt: str) -> None:
    with open(sh_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'RESUME_FROM="[^"]*"', f'RESUME_FROM="{ckpt}"', content)
    with open(sh_path, "w", encoding="utf-8") as f:
        f.write(content)

_set_resume_from("scripts/train_vqvae.sh", vqvae_ckpt if have_vqvae else "")
_set_resume_from("scripts/train_ldm.sh", ldm_ckpt if have_ldm else "")
print(f"  [续训] train_vqvae.sh RESUME_FROM -> {vqvae_ckpt if have_vqvae else '(空，从零训练)'}")
print(f"  [续训] train_ldm.sh    RESUME_FROM -> {ldm_ckpt if have_ldm else '(空，从零训练)'}")

# ===== 6. GPU 软检测 + 按实例自动调优训练/推理参数 =====
print("\n===== 硬件检测与性能调优 =====")
import torch

cpu_cores = os.cpu_count() or 4

# --- 内存感知的 workers 计算（Cloud Studio 小内存实例防 OOM）---
def _avail_mem_gb() -> float:
    """读取容器 cgroup 内存上限；失败则回退到 /proc/meminfo"""
    try:
        with open("/sys/fs/cgroup/memory.max", "r", encoding="utf-8") as f:
            v = f.read().strip()
            if v.isdigit() and int(v) > 0:
                return int(v) / 1024 ** 3
    except Exception:
        pass
    try:
        with open("/proc/meminfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return float(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return 8.0  # 保守默认

avail_mem_gb = _avail_mem_gb()
DL_WORKERS = max(2, min(8, cpu_cores))       # DataLoader 进程数：与 CPU 并行匹配，喂饱 GPU
RENDER_WORKERS = max(2, min(16, cpu_cores))  # 字形渲染线程数：数据准备阶段压榨 CPU 核数

# 小内存实例降级并行度，防止渲染字形时内存超限导致内核被杀（"运行中止"）
if avail_mem_gb < 4:
    RENDER_WORKERS = 2
    DL_WORKERS = 2
elif avail_mem_gb < 8:
    RENDER_WORKERS = min(4, RENDER_WORKERS)
    DL_WORKERS = min(2, DL_WORKERS)
print(f"  可用内存: {avail_mem_gb:.1f} GB（workers 已按内存限制适配）")

gpu_name, vram_gb = None, 0.0
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = prop.total_memory / 1024**3
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB) | CUDA {torch.version.cuda} | PyTorch {torch.__version__}")
else:
    print("未检测到 GPU：当前实例为 CPU 规格")
    print("  → 可运行: Cell 2（数据准备）/ Cell 6-8（SVG 转换与打包下载）")
    print("  → 不可运行: Cell 3/4（训练）/ Cell 5（推理），请切换 GPU 实例后重跑 Cell 0 + Cell 1")

profile = HW_PROFILE
if profile == "auto":
    if gpu_name is None:
        profile = "cpu"
    elif any(k in gpu_name.upper() for k in ("V100", "A10", "L40", "L20", "A100", "H100")):
        profile = "v100"    # 高带宽档
    elif "T4" in gpu_name.upper():
        profile = "t4"
    elif vram_gb >= 15:
        profile = "v100"    # 未知型号但显存充足，按高带宽档尝试
    else:
        profile = "t4"

# 各档位推荐参数（以 16GB 显存为基准设计）
# v100 档（V100/A10/L40 等，带宽 ≥600GB/s）：加大 batch 吃满带宽吞吐
# t4   档（T4，320GB/s，16GB）：按已知安全值配置
if profile == "v100":
    VQVAE_BS, LDM_BS, EVAL_BS, INFER_BS = 48, 96, 16, 32
elif profile == "t4":
    VQVAE_BS, LDM_BS, EVAL_BS, INFER_BS = 44, 64, 8, 16
else:  # cpu 档：训练/推理参数不动（本实例不执行），仅调数据准备并行度
    VQVAE_BS = LDM_BS = EVAL_BS = INFER_BS = None

print(f"\n性能档位: {HW_PROFILE} → {profile} | CPU 核数: {cpu_cores}")
print(f"  DataLoader workers = {DL_WORKERS}（GPU 并发与 CPU 并行匹配，避免 GPU 等数据空转）")
print(f"  字形渲染 workers   = {RENDER_WORKERS}（数据准备阶段并行渲染）")
if profile != "cpu":
    print(f"  VQ-VAE batch = {VQVAE_BS} | LDM batch = {LDM_BS} | 评估 batch = {EVAL_BS} | 推理 batch = {INFER_BS}")
    print("  （若训练 OOM：把 HW_PROFILE 改为 \"t4\" 或手动调低 scripts/*.sh 中 BATCH_SIZE 后重跑本 Cell）")

def set_sh_var(sh_path: str, var: str, value) -> None:
    """改写 sh 脚本中的 VAR=... 行（保留行尾注释）"""
    with open(sh_path, encoding="utf-8") as f:
        lines = f.readlines()
    for i, ln in enumerate(lines):
        if ln.startswith(var + "="):
            body = ln[len(var) + 1:].rstrip("\n")
            tail = ""
            if "#" in body:
                tail = "  " + body[body.index("#"):]
            lines[i] = f"{var}={value}{tail}\n"
            break
    with open(sh_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

# --- 数据准备（任何实例都生效）---
set_sh_var("scripts/prepare_dataset.sh", "NUM_WORKERS", RENDER_WORKERS)
# extract_charset 仅做数据集划分，无 GPU 时切 cpu 避免异常
set_sh_var("scripts/extract_charset.sh", "DEVICE", '"cuda"' if gpu_name else '"cpu"')

# --- 训练 / 推理（仅 GPU 档位时改写）---
if profile != "cpu":
    set_sh_var("scripts/train_vqvae.sh", "BATCH_SIZE", VQVAE_BS)
    set_sh_var("scripts/train_vqvae.sh", "NUM_WORKERS", DL_WORKERS)
    set_sh_var("scripts/train_ldm.sh", "BATCH_SIZE", LDM_BS)
    set_sh_var("scripts/train_ldm.sh", "NUM_WORKERS", DL_WORKERS)
    set_sh_var("scripts/train_ldm.sh", "EVAL_BATCH_SIZE", EVAL_BS)
    set_sh_var("scripts/inference.sh", "BATCH_SIZE", INFER_BS)
    set_sh_var("scripts/compute_metrics.sh", "EVAL_BATCH_SIZE", EVAL_BS)
    print("\n脚本参数已按硬件档位自动调优（BATCH_SIZE / NUM_WORKERS / EVAL_BATCH_SIZE）")

print(f"\n全部初始化完成！字体: {font_path}")

---
# 🖥️ 阶段一：数据准备 —— 选择最便宜的 CPU 实例即可

> **实例建议**：平台最便宜的 CPU 规格（1核2G 起步；2核4G 安装依赖更快）。此阶段为纯 CPU 的文件管理与字体渲染，**无需 GPU**，不要浪费 GPU 机时。
>
> 若当前已在 GPU 实例（如续训场景），也可直接运行，非强制。

---
## Cell 2: 数据准备（约 10-20 分钟，断连可恢复）

> 分析字体覆盖率 → 渲染字形图片（并行度已按 CPU 核数自动调整）→ 提取训练/验证字符集
>
> **幂等**：已完成（且字体未变更）会自动跳过；**更换字体后会自动清空旧 `data/` 并重建**，杜绝新旧字体数据混合。

In [ ]:
import os, json, subprocess, sys

if not DO_DATA_PREP:
    print("DO_DATA_PREP=False，跳过 Cell 2")
else:
    def _load_state() -> dict:
        if os.path.exists(STATE_FILE):
            try:
                with open(STATE_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save_state(s: dict) -> None:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)

    def _run_script(script_path: str, step_name: str) -> None:
        """运行 shell 脚本，失败时打印完整 stderr 后抛出异常"""
        print(f"\n{'='*60}")
        print(f"  执行: {step_name}")
        print(f"  脚本: {script_path}")
        print(f"{'='*60}")
        result = subprocess.run(["bash", script_path], capture_output=True, text=True)
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            if result.stderr:
                print("[STDERR 如下  ↓↓↓ ]", file=sys.stderr)
                print(result.stderr, file=sys.stderr)
            raise RuntimeError(
                f"{step_name} 失败！返回码: {result.returncode}\n"
                f"请查看上方 STDERR 定位具体原因。"
            )
        print(f"[OK] {step_name} 完成")

    state = _load_state()

    data_done   = os.path.isdir("data/reference") and os.path.isdir("data/target")
    splits_done = os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt") and \
                  os.path.exists(f"charsets/splits/{FONT_NAME}/val.txt")
    same_font   = state.get("data_font") == FONT_NAME   # 数据集必须与当前字体绑定

    if state.get("data_prep_done") and same_font and data_done and splits_done:
        print(f"字体 {FONT_NAME} 的数据准备已完成，跳过 Cell 2")
    else:
        if not same_font and state.get("data_font"):
            print(f"[字体切换] {state['data_font']} → {FONT_NAME}：将清空 data/ 并重新生成")
        _run_script("scripts/analyze_font.sh",    "Step 1/3: 分析字体覆盖率")
        _run_script("scripts/prepare_dataset.sh", "Step 2/3: 生成数据集图片")
        _run_script("scripts/extract_charset.sh", "Step 3/3: 提取训练/验证字符集")

        if not (os.path.isdir("data/reference") and os.path.isdir("data/target")):
            raise RuntimeError("data/ 目录未正确生成。请查看上方 STDERR 输出。")
        if not os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt"):
            raise RuntimeError("train.txt 未生成。请查看上方 STDERR 输出。")

        state["data_prep_done"] = True
        state["data_font"] = FONT_NAME   # 记录数据集归属字体
        _save_state(state)
        print("\n===== 数据准备完成，已记录字体绑定状态 =====")

---
# 🚀 阶段二：模型训练 —— 请切换到 GPU 实例（首选 V100）

> **实例建议**：
> - **首选 V100**：900 GB/s 显存带宽。训练需成千上万次加噪/去噪循环、高频读写海量潜在特征数据，V100 让数据"秒到"，显卡算个不停
> - 次选 A10（600 GB/s+，性能均衡，性价比不如 V100）；L40 性能过剩且最贵（8 机时/时），不推荐
> - T4（320 GB/s）最便宜，但大部分时间在等数据，总训练时长明显拉长——仅预算极度敏感时使用
>
> ⚠️ **切换实例后先重跑 Cell 0 + Cell 1**（重新检测 GPU 型号、自动调优 `BATCH_SIZE` / `NUM_WORKERS`），再从 Cell 3 继续。

---
## Cell 3: 训练 VQ-VAE（约 6-8 小时，自动精确续训）

> **断连恢复**：重跑 Cell 0、Cell 1（自动改写 `RESUME_FROM`），再运行本 Cell 即从断点 epoch 精确续训。
>
> 本 Cell 开头含 **GPU 硬断言**——在 CPU 实例运行会直接提示切换实例。

In [ ]:
import os, subprocess, torch

if not DO_TRAIN_VQVAE:
    print("DO_TRAIN_VQVAE=False，跳过 Cell 3")
else:
    assert torch.cuda.is_available(), (
        "Cell 3 需要 GPU 实例（推荐 V100）。请在工作空间切换到 GPU 实例后，"
        "重跑 Cell 0 + Cell 1（自动调优训练参数），再运行本 Cell"
    )
    assert os.path.isdir("data/target"), "data/ 不存在，请先运行 Cell 2 完成数据准备"

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("训练参数已由 Cell 1 按当前硬件调优（含 AMP 混合精度 + DataLoader 并行）")
    subprocess.run(["bash", "scripts/train_vqvae.sh"], check=True)

---
## Cell 4-前置操作：离线放置 VGG16 权重（LDM 训练与指标前置）

> **为什么需要**：Cell 4（训练 LDM）与 Cell 5（LPIPS 指标）都会用到 VGG16 权重。该权重约 553MB，在云环境从官网下载很慢，故提前离线放置。
>
> **操作（必做）**：先手动把 `vgg16-397923af.pth` 上传到腾讯云工作空间（推荐根目录 `/workspace/`）。本 Cell 运行后会自动搜索并移动到 PyTorch Hub 缓存目录 `/root/.cache/torch/hub/checkpoints/`，后续 Cell 4 / Cell 5 直接命中、免下载。
>
> ⚠️ 注意：此文件是 **VGG 预训练权重**，不是 `checkpoints/vqvae_{FONT}.pth`（那是本项目的 VQ-VAE 模型），两者**不要混淆、不要放进 `checkpoints/`**。
>
> 📥 **本地文件丢失时下载备份**（上传至腾讯云用）：
> - GitHub 仓库备份：`https://github.com/ICW-k/HanziGen_ICWfork/raw/main/vgg16-397923af.pth`
> - 或从 PyTorch Hub 重新获取（较慢）：`torch.hub.load('pytorch/vision:v0.9.0', 'vgg16', pretrained=True)`
>
> 若你已联网且不在乎下载时间，也可跳过本 Cell（Cell 4 / Cell 5 会自动在线拉取）。
---


In [ ]:
import os, shutil

# 方案 1：自动搜索文件（最稳妥）
def find_file(filename, search_roots=("/workspace", "/home", "/root", os.getcwd(), os.path.dirname(os.getcwd()))):
    for root in search_roots:
        if os.path.isdir(root):
            for dirpath, dirnames, filenames in os.walk(root):
                if filename in filenames:
                    return os.path.join(dirpath, filename)
    return None

src = find_file("vgg16-397923af.pth")
if src is None:
    print("❌ 还是没找到，请确认文件已上传到工作空间（推荐放到根目录）")
    print(f"当前工作目录: {os.getcwd()}")
else:
    # 放在 PyTorch Hub 缓存目录，lpips 加载 VGG 时会直接命中，无需联网下载
    dst_dir = "/root/.cache/torch/hub/checkpoints"
    dst = os.path.join(dst_dir, "vgg16-397923af.pth")
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.move(src, dst)
    print(f"✅ VGG16 权重已就位: {dst}")
    print("   Cell 5 的 LPIPS 指标将直接使用本文件，跳过远程下载")

# 方案 2（备选）：若你明确知道文件就在根目录，直接用绝对路径，注释掉上方、启用下方：
# src = "/workspace/vgg16-397923af.pth"  # 或 /home/jovyan/vgg16-397923af.pth
# dst = "/root/.cache/torch/hub/checkpoints/vgg16-397923af.pth"
# os.makedirs(os.path.dirname(dst), exist_ok=True)
# shutil.move(src, dst); print(f"✅ 已移动到: {dst}")

---
## Cell 4: 训练 LDM（约 10-15 小时，自动精确续训）

> 依赖 Cell 3 产物 `checkpoints/vqvae_{FONT_NAME}.pth`。**仍使用同一 GPU 实例，无需切换**。
>
> LDM 在 64×64 潜在空间训练、显存占用小，`BATCH_SIZE` 已按带宽档自动调大（V100 档 96 / T4 档 64），配合 8 进程 DataLoader + 预取（prefetch）充分压榨 GPU 吞吐。
>
> **断连恢复**：重跑 Cell 0、Cell 1，再运行本 Cell 即从断点 epoch 精确续训。

In [ ]:
import os, subprocess, torch

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"

if not DO_TRAIN_LDM:
    print("DO_TRAIN_LDM=False，跳过 Cell 4")
else:
    assert torch.cuda.is_available(), (
        "Cell 4 需要 GPU 实例（推荐 V100）。请在工作空间切换到 GPU 实例后，"
        "重跑 Cell 0 + Cell 1（自动调优训练参数），再运行本 Cell"
    )
    assert os.path.exists(vqvae_ckpt), f"{vqvae_ckpt} 不存在，请先完成 Cell 3 训练 VQ-VAE"

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("VQ-VAE 检查点确认：")
    !ls -lh checkpoints/
    subprocess.run(["bash", "scripts/train_ldm.sh"], check=True)

# 🎨 阶段三A：推理生成 + 指标 —— GPU 实例（追求省钱可从 V100 换到 T4）

> **实例建议**：推理是"逐批计算"，对显存带宽的敏感度远低于训练——**T4（最便宜的 GPU）即可胜任**；嫌切换麻烦也可继续用 V100。
>
> GBK 基准 13,000+ 缺字建议预留 30-60 分钟 GPU 机时。
>
> ⚠️ 若换了实例：先重跑 Cell 0 + Cell 1（会自动调优推理 `BATCH_SIZE`）。

---


## Cell 5: 推理生成 + 评估指标

> 补字基准在 Cell 0 的 `CHARSET_BASE` 选择：`jf7000`（默认）/ `unihan` / `gbk`（简体 20,902 字）/ `gb2312`（核心 6,763 字）。
>
> 依赖 `checkpoints/ldm_{FONT_NAME}.pth` 存在。本 Cell 开头含 **GPU 硬断言**。

In [ ]:
import os, re, subprocess, torch

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not DO_INFERENCE:
    print("DO_INFERENCE=False，跳过 Cell 5")
else:
    assert torch.cuda.is_available(), (
        "Cell 5 需要 GPU 实例（推理用 T4 即可）。请在工作空间切换到 GPU 实例后，"
        "重跑 Cell 0 + Cell 1，再运行本 Cell"
    )
    assert os.path.exists(ldm_ckpt), f"{ldm_ckpt} 不存在，请先完成 Cell 4 训练 LDM"

    # ===== 1. 按补字基准确定字符集文件 =====
    def build_charset_path(base: str) -> str:
        if base in ("gbk", "gb2312"):
            # 用系统编码器直接计算 GBK / GB2312 缺失字，无需额外字集文件
            from fontTools.ttLib import TTFont
            base_chars = set()
            for cp in range(0x4E00, 0xA000):
                try:
                    chr(cp).encode(base)
                    base_chars.add(chr(cp))
                except UnicodeEncodeError:
                    pass
            font = TTFont(f"fonts/{TARGET_FONT}", fontNumber=0)
            cmap = set()
            for table in font["cmap"].tables:
                if table.isUnicode():
                    cmap.update(table.cmap.keys())
            missing = sorted(base_chars - {chr(c) for c in cmap})
            out_dir = f"charsets/{base}_coverage/{FONT_NAME}"
            os.makedirs(out_dir, exist_ok=True)
            out_path = f"{out_dir}/missing.txt"
            with open(out_path, "w", encoding="utf-8") as f:
                f.write("\n".join(missing))
            print(f"[{base}] 基准 {len(base_chars)} 字，字体缺失 {len(missing)} 字 → {out_path}")
            return out_path
        if base == "unihan":
            p = f"charsets/unihan_coverage/{FONT_NAME}/missing.txt"
        else:  # jf7000
            p = f"charsets/jf7000_coverage/{FONT_NAME}/missing.txt"
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} 不存在，请先运行 Cell 2 完成字体分析")
        return p

    charset_path = build_charset_path(CHARSET_BASE)
    print(f"补字基准: {CHARSET_BASE} → {charset_path}")

    # ===== 2. inference.sh 指向该字符集 =====
    with open("scripts/inference.sh", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'CHARSET_PATH="[^"]*"', f'CHARSET_PATH="{charset_path}"', content)
    content = re.sub(r'DEVICE="[^"]*"', 'DEVICE="cuda"', content)
    with open("scripts/inference.sh", "w", encoding="utf-8") as f:
        f.write(content)
    print("inference.sh 已指向补字基准字符集")

    # ===== 3. 推理 + 指标 =====
    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)
    print("\n===== 计算评估指标 =====")
    subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)

    print("\n===== GPU 阶段全部完成 =====")
    print(f"生成 PNG 位置: samples_{FONT_NAME}/inference/gen/")
    print("\n下一步：SVG 向量化为纯 CPU 任务，建议切换到 16核32G CPU 实例运行 Cell 6-8，释放 GPU 节省机时")
    print("（切换后重跑 Cell 0 恢复变量即可；Cell 1 可选，自检幂等）")

---
# 📦 阶段三B：SVG 向量化 + 打包下载 —— 请切换到 CPU 实例（首选 16核32G）

> **实例建议**：
> - **首选 16核32G**（2 机时/时）：写入 1万+ 张 10KB 小文件，瓶颈在内存缓存，32G 内存能有效缓存文件索引、避免磁盘频繁寻址卡顿
> - 省钱备选 8核16G（1 机时/时）；**千万不要选 8G 以下内存，否则会因"内存不足"直接崩溃**
> - 本阶段不依赖显卡（纯 CPU）——**可释放 GPU 实例节省机时**
>
> ⚠️ 切换实例后：**重跑 Cell 0**（恢复变量）即可运行 Cell 6-8；Cell 1 可跑可不跑（自检幂等，不跑也不影响本阶段）。

---
## Cell 6: 转换 SVG + 产出总览

In [ ]:
import os, subprocess

png_dir = f"samples_{FONT_NAME}/inference/gen"
svg_dir = f"svgs_{FONT_NAME}"

if not os.path.isdir(png_dir):
    print(f"{png_dir} 不存在，请先在 GPU 实例完成 Cell 5 推理")
else:
    print("本 Cell 为纯 CPU 任务，无需 GPU")
    print("\n===== 转换 SVG（1万+ 文件约需 10-30 分钟，取决于 CPU 规格）=====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)

    print("\n===== 产出总览 =====")
    n_png = len([f for f in os.listdir(png_dir) if f.endswith(".png")])
    n_svg = len([f for f in os.listdir(svg_dir) if f.endswith(".svg")]) if os.path.isdir(svg_dir) else 0
    print(f"生成 PNG: {n_png} 张 → {png_dir}/")
    print(f"转换 SVG: {n_svg} 个 → {svg_dir}/")

    print("\n[checkpoints]")
    if os.path.isdir("checkpoints"):
        for f in sorted(os.listdir("checkpoints")):
            size = os.path.getsize(os.path.join("checkpoints", f)) / 1024**2
            print(f"  {f}  ({size:.1f} MB)")
    else:
        print("  无")

    print(f"\n[samples_{FONT_NAME}]")
    if os.path.isdir(f"samples_{FONT_NAME}"):
        for d in sorted(os.listdir(f"samples_{FONT_NAME}")):
            print(f"  {d}/")
    else:
        print("  无")

---
## Cell 7: 导出并下载 SVG 结果

> 把 `svgs_{FONT_NAME}/` 打包成 zip → 浏览器下载链接 + 同步保存到工作空间根目录（文件树可见处）+ 页面内预览前 9 个。
>
> 下载后可用 **FontForge**：打开原字体 → Import 这些 SVG → Generate 导出完整字体。

In [ ]:
import os, glob, zipfile, io, base64
from IPython.display import HTML, display

svg_dir = f"svgs_{FONT_NAME}"
if not os.path.isdir(svg_dir):
    print(f"{svg_dir} 不存在，请先运行 Cell 6 完成转换")
else:
    svg_files = sorted(glob.glob(os.path.join(svg_dir, "*.svg")))
    print(f"共找到 {len(svg_files)} 个 SVG 文件")

    # 1. 打包成 zip（内存中生成）
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in svg_files:
            zf.write(f, arcname=os.path.basename(f))
    zip_data = zip_buffer.getvalue()
    zip_name = f"svgs_{FONT_NAME}.zip"

    # 2. 同步保存到工作空间根目录（fonts 所在目录，文件树可见）
    saved_paths = []
    roots = [os.path.abspath(os.getcwd()), os.path.dirname(os.path.abspath(os.getcwd())),
             "/workspace", "/home", "/root"]
    seen = set()
    for r in roots:
        if r and r not in seen and os.path.isdir(r):
            seen.add(r)
            try:
                p = os.path.join(r, zip_name)
                with open(p, "wb") as f:
                    f.write(zip_data)
                saved_paths.append(p)
            except Exception:
                pass
    for p in saved_paths:
        print(f"  [已保存] {p} ({len(zip_data)/1024:.0f} KB)")

    # 3. 浏览器内点击下载（base64 直链；文件过大时优先从工作空间根目录下载）
    b64 = base64.b64encode(zip_data).decode()
    download_link = (
        '<a href="data:application/zip;base64,' + b64 + f'" download="{zip_name}" '
        'style="display:inline-block;font-size:18px;font-weight:bold;color:#fff;'
        'background:#1a73e8;padding:12px 28px;border-radius:8px;'
        'text-decoration:none;">'
        f'下载全部 SVG ({len(svg_files)} 个 / {len(zip_data)/1024:.0f} KB)</a>'
    )
    display(HTML(download_link))

    # 4. 页面内预览前 9 个 SVG
    cards = []
    for f in svg_files[:9]:
        with open(f, encoding="utf-8") as fh:
            svg = fh.read()
        svg = svg.replace("<svg ", '<svg width="80" height="80" style="background:#fff;" ', 1)
        name = os.path.splitext(os.path.basename(f))[0]
        cards.append(
            f'<div style="border:1px solid #e0e0e0;border-radius:8px;padding:8px;'
            f'text-align:center;width:96px;">{svg}<div style="font-size:12px;color:#555;">{name}</div></div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px;">' + "".join(cards) + "</div>"))

---
## Cell 8: 导出全部生成结果（PNG + SVG）为 zip

> Cell 7 只打包 SVG，本 Cell 把**全部产出**一起打包，方便本地用其他工具（如 SVG → OTF 脚本）装配字体：
>
> - `svgs_{FONT_NAME}/` — 全部 SVG 矢量字形
> - `samples_{FONT_NAME}/inference/` — 生成的 PNG（gen=生成, gt=真值, ref=参考）
> - `samples_{FONT_NAME}/val/`、`train/` — 训练/验证样本对比图
>
> 不需要的目录可在 `include_dirs` 列表自行增删。

In [ ]:
import os, glob, zipfile, io, base64
from IPython.display import HTML, display

FONT = FONT_NAME

# ===== 要打包的目录（按需增删） =====
include_dirs = [
    f"svgs_{FONT}",                          # SVG 矢量字形
    f"samples_{FONT}/inference/gen",          # 生成的 PNG
    f"samples_{FONT}/inference/gt",           # 真值 PNG
    f"samples_{FONT}/inference/ref",          # 参考 PNG
    f"samples_{FONT}/val",                    # 验证样本
    f"samples_{FONT}/train",                  # 训练样本
]

zip_name = f"hanzigen_output_{FONT}.zip"
buf = io.BytesIO()
total_files = 0
found_any = False

with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        if not os.path.isdir(folder):
            print(f"  [跳过] {folder} 不存在")
            continue
        found_any = True
        files = [
            p for p in glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)
            if os.path.isfile(p)
        ]
        for f in files:
            arcname = os.path.relpath(f)
            zf.write(f, arcname=arcname)
        print(f"  [OK] {folder} → {len(files)} 个文件")
        total_files += len(files)

if not found_any:
    print("所有目录都不存在，请先运行 Cell 5 / Cell 6 生成结果")
else:
    zip_data = buf.getvalue()

    # 1. 同步保存到工作空间根目录（文件树可见处）
    saved_paths = []
    roots = [os.path.abspath(os.getcwd()), os.path.dirname(os.path.abspath(os.getcwd())),
             "/workspace", "/home", "/root"]
    seen = set()
    for r in roots:
        if r and r not in seen and os.path.isdir(r):
            seen.add(r)
            try:
                p = os.path.join(r, zip_name)
                with open(p, "wb") as f:
                    f.write(zip_data)
                saved_paths.append(p)
            except Exception:
                pass
    print(f"\n共 {total_files} 个文件，{len(zip_data)/1024/1024:.1f} MB")
    for p in saved_paths:
        print(f"  [已保存] {p}")

    # 2. 浏览器内点击下载（文件过大时优先从工作空间根目录下载）
    b64 = base64.b64encode(zip_data).decode()
    download_link = (
        '<a href="data:application/zip;base64,' + b64 + f'" download="{zip_name}" '
        'style="display:inline-block;font-size:18px;font-weight:bold;color:#fff;'
        'background:#188038;padding:12px 28px;border-radius:8px;'
        'text-decoration:none;">'
        f'下载全部结果 ({total_files} 个文件 / {len(zip_data)/1024/1024:.1f} MB)</a>'
    )
    display(HTML(download_link))

---
## 断连 / 换实例恢复指南

Cloud Studio 工作空间存储是持久化的（checkpoints / data / samples 不会丢失），恢复只需按序重跑：

| 情况 | 操作 |
|---|---|
| 同一实例重启会话 | Cell 0 → Cell 1 → 依次 Cell 2/3/4/5（已完成阶段自动跳过，未完成的训练从最近 checkpoint 精确续训） |
| 切换实例（CPU↔GPU / V100→T4） | **Cell 0 → Cell 1**（重新检测硬件并自动调优参数）→ 从目标阶段继续 |
| 训练完换 CPU 实例导出 | **Cell 0**（恢复变量）→ Cell 6 → Cell 7/8（Cell 1 可选） |

> **无需手动编辑脚本**：Cell 1 已自动处理 `RESUME_FROM` 与全部性能参数。

---

## 更换新字体训练（数据自动隔离说明）

工作流支持"上传新字体 → Cell 0 改字体名 → 从 Cell 1 开始一键训练"，隔离机制如下：

1. **训练数据（`data/`）**：该目录全局唯一、不分字体。为此 Cell 1/Cell 2 把数据集状态与字体名绑定（`colab_state.json` 的 `data_font` 字段）：
   - Cell 1 检测到 `TARGET_FONT` 变更时，自动作废旧的数据准备标记；
   - Cell 2 因此重新执行 `prepare_dataset.py`，其内置 `shutil.rmtree(data)` 会**先清空旧字体图像再渲染新字体**，杜绝新旧字体数据混合。
2. **字符集（`charsets/`）**：按字体名天然隔离（`jf7000_coverage/{font}/`、`unihan_coverage/{font}/`、`splits/{font}/`）。
3. **模型与产出**：`checkpoints/vqvae_{font}.pth`、`ldm_{font}.pth`、`samples_{font}/`、`svgs_{font}/` 均按字体名隔离，互不覆盖。
4. **续训安全**：Cell 1 每次运行都把 `RESUME_FROM` 重写为**当前字体**的检查点（无则清空），避免 A 字体的权重被误用于 B 字体。

> 唯一需人工干预的情况：**同名更新**字体文件（文件名不变、内容变化）时，请手动删除该字体的 `checkpoints/vqvae_{font}.pth` 与 `ldm_{font}.pth` 从零训练。

---

## 常见问题

| 问题 | 解决 |
|------|------|
| `fonts/xxx.ttf` 不存在 | Cell 0 填的字体名和 `fonts/` 里的文件名是否一致（区分大小写） |
| CPU 实例跑 Cell 1 报 GPU 错误 | 新版已改为软检测不会报错；Cell 3/4/5 才有硬断言 |
| 切换实例后 BATCH_SIZE 没变 | 重跑 Cell 0 + Cell 1 触发硬件自动调优 |
| 训练 OOM | Cell 0 把 `HW_PROFILE` 改为 `"t4"`，或手动调低 `scripts/train_*.sh` 的 `BATCH_SIZE`，重跑 Cell 1 |
| 换新字体训练 | 直接 Cell 0 改名即可，旧字体数据自动清空重建（见上方说明） |
| 当前字体想从头重训 | 删除 `checkpoints/vqvae_{font}.pth`、`ldm_{font}.pth` 与 `colab_state.json`，重跑 Cell 1 |
| 只想补 GBK 简体缺字 | Cell 0 设 `CHARSET_BASE="gbk"`（训练完成后只跑 Cell 5-8 即可）；也可使用仓库中的「GBK 补字推理版」notebook |
| 生成太慢 | `scripts/inference.sh` 的 `SAMPLE_STEPS=50` 改为 `20`（速度约 2.5 倍，质量略降） |
| SVG 阶段内存不足崩溃 | 换 16核32G 实例；8G 以下内存必崩 |
| 工作空间长期不用被回收 | 定期下载 `checkpoints/` 和 `svgs_*/` 到本地备份 |